Libraries

# **Google Mount**

In [1]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)
print("Mounted:", os.path.exists('/content/drive/MyDrive'))

Mounted at /content/drive
Mounted: True


In [2]:
# ── CELL 1: Verify GPU ────────────────────────────────────────────────────────
# Expected: Tesla T4, ~15 GB VRAM
# If you see K80 or < 14 GB: Runtime → Disconnect and delete runtime → reconnect
!nvidia-smi
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    icon = '✅' if vram >= 14 else '⚠️ '
    print(f'\n{icon} GPU: {name}  ({vram:.0f} GB VRAM)')
    if vram < 14:
        print('   You have a K80 (12 GB). Reconnect to get a T4 (16 GB).')
else:
    print('\n❌ No GPU — go to Runtime → Change runtime type → T4 GPU')

Sun Sep 13 02:48:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# **Commit**

In [5]:
# # ── CELL 1: Setup — run this ONCE per Colab session ─────────────────────────
import subprocess
import os
import shutil
from google.colab import userdata

PAT = userdata.get('GITHUB_PAT')
REPO_URL = f'https://{PAT}@github.com/Break-Through-Tech/Automation-Anywhere-1B-domain-specific-theme-labeling-via-slm-distillation.git'
REPO = '/content/project'
BRANCH = 'btt_setup_VD'


def setup_repo():
    """Clone the repo fresh if it doesn't exist yet, otherwise just pull
    the latest changes for my branch."""
    if not os.path.exists(f'{REPO}/.git'):
        # not cloned yet this session (or it's broken) — clone fresh
        if os.path.exists(REPO):
            print("Removing broken non-git folder...")
            subprocess.run(['rm', '-rf', REPO])
        print(f"Cloning branch '{BRANCH}' ...")
        r = subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO],
                            capture_output=True, text=True)
        print(r.stdout, r.stderr)
    else:
        print(f"Repo already exists — pulling latest '{BRANCH}' ...")
        subprocess.run(['git', 'checkout', BRANCH], cwd=REPO)
        r = subprocess.run(['git', 'pull', 'origin', BRANCH],
                            capture_output=True, text=True, cwd=REPO)
        print(r.stdout, r.stderr)

    print("Is git repo:", os.path.exists(f'{REPO}/.git'))

    # set git identity once (safe to re-run)
    subprocess.run(['git', 'config', '--global', 'user.email', 'vd35@rice.edu'], cwd=REPO)
    subprocess.run(['git', 'config', '--global', 'user.name', 'vantastics'], cwd=REPO)


def detect_code_dir():
    """Some branches put main.py/configs at repo root, others under code/.
    Auto-detect which one this branch uses, same logic as 00_session_setup.ipynb."""
    if os.path.exists(f'{REPO}/main.py'):
        return REPO
    elif os.path.exists(f'{REPO}/code/main.py'):
        return f'{REPO}/code'
    else:
        raise FileNotFoundError(
            f"Could not find main.py at {REPO}/main.py or {REPO}/code/main.py — "
            f"check repo contents: {os.listdir(REPO)}"
        )


setup_repo()
CODE_DIR = detect_code_dir()
print(f"Code directory: {CODE_DIR}")

# make my personal config copy — only if it doesn't already exist,
# so re-running this cell never overwrites my own edits
personal_config = f'{CODE_DIR}/configs/phase1_config_vd.yaml'
shared_config = f'{CODE_DIR}/configs/phase1_config.yaml'
if not os.path.exists(personal_config):
    shutil.copy(shared_config, personal_config)
    print(f"Created your personal config: {personal_config}")
else:
    print(f"Personal config already exists, leaving it as-is: {personal_config}")

Cloning branch 'btt_setup_VD' ...
 Cloning into '/content/project'...

Is git repo: True
Code directory: /content/project/code
Personal config already exists, leaving it as-is: /content/project/code/configs/phase1_config_vd.yaml


In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{CODE_DIR}/requirements.txt'],
    capture_output=True, text=True
)
print(result.stdout[-2000:], result.stderr[-2000:])

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{CODE_DIR}/requirements_colab.txt'],
    capture_output=True, text=True
)
print(result.stdout[-2000:], result.stderr[-2000:])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.3 MB/s eta 0:00:00
 


In [ ]:
# ── CELL 2: Reusable commit helper — run setup_repo() first, then call this
#            any time I want to push a file to my branch.
import subprocess
import os
import shutil


def commit_file(source_path: str, dest_relative_path: str, commit_message: str,
                 under_code_dir: bool = True):
    """
    Copy a file into the repo and push it to your branch.

    source_path         — full path to the file right now (e.g. in Drive)
    dest_relative_path  — where it should live inside the repo (or code dir),
                           e.g. 'notebooks/fine_tuning_VD.ipynb'
    commit_message       — your commit message
    under_code_dir       — True if this path is relative to CODE_DIR (e.g.
                           notebooks/, configs/ — most things). False if it's
                           relative to the REPO root instead.
    """
    base_dir = CODE_DIR if under_code_dir else REPO
    dest_path = f'{base_dir}/{dest_relative_path}'
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)

    shutil.copy(source_path, dest_path)
    print(f"Copied to: {dest_path}")

    # git commands always run relative to REPO root, so the path passed to
    # `git add` needs to include the code/ prefix if that's where the file is
    git_relative_path = os.path.relpath(dest_path, REPO)

    for cmd in [
        ['git', 'checkout', BRANCH],
        ['git', 'add', git_relative_path],
        ['git', 'commit', '-m', commit_message],
        ['git', 'push', 'origin', BRANCH],
    ]:
        r = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO)
        out = (r.stdout + r.stderr).strip()
        if out:
            print(out)

    print('\nDone. Verify at:')
    print(f'https://github.com/Break-Through-Tech/Automation-Anywhere-1B-domain-specific-theme-labeling-via-slm-distillation/tree/{BRANCH}/{os.path.dirname(git_relative_path)}')


# ── Example usage ────────────────────────────────────────────────────────
# Every time you want to save a notebook (or any file) to your branch,
# just call this one line — no need to repeat the clone/copy/commit steps:

commit_file(
    source_path='/content/drive/MyDrive/Colab Notebooks/fine_tuning_VD.ipynb',
    dest_relative_path='notebooks/fine_tuning_VD.ipynb',
    commit_message='Update fine tuning notebook',
    under_code_dir=False,   # notebooks/ lives at repo root, not under code/
)


In [ ]:
#Conflict (delete on Git, upload from local)
# import subprocess
# REPO = '/content/project'
# BRANCH = 'btt_setup_VD'

# subprocess.run(['git', 'config', 'pull.rebase', 'false'], cwd=REPO)

# r = subprocess.run(['git', 'pull', 'origin', BRANCH], capture_output=True, text=True, cwd=REPO)
# print(r.stdout, r.stderr)

# # keep your local version (the file existing, with your fix)
# r = subprocess.run(['git', 'add', 'code/run_experiments.py'], capture_output=True, text=True, cwd=REPO)
# print(r.stdout, r.stderr)

# r = subprocess.run(['git', 'commit', '-m', 'Merge — keep run_experiments.py'], capture_output=True, text=True, cwd=REPO)
# print(r.stdout, r.stderr)

# r = subprocess.run(['git', 'push', 'origin', BRANCH], capture_output=True, text=True, cwd=REPO)
# print(r.stdout, r.stderr)

CONFLICT (modify/delete): code/run_experiments.py deleted in 9e43b373d1cf0fed3fb8d1cb4b91db39e1219379 and modified in HEAD.  Version HEAD of code/run_experiments.py left in tree.
Automatic merge failed; fix conflicts and then commit the result.
 From https://github.com/Break-Through-Tech/Automation-Anywhere-1B-domain-specific-theme-labeling-via-slm-distillation
 * branch            btt_setup_VD -> FETCH_HEAD
   d297463..9e43b37  btt_setup_VD -> origin/btt_setup_VD

 
[btt_setup_VD 4328463] Merge — keep run_experiments.py
 
 To https://github.com/Break-Through-Tech/Automation-Anywhere-1B-domain-specific-theme-labeling-via-slm-distillation.git
   9e43b37..4328463  btt_setup_VD -> btt_setup_VD



In [ ]:
#Commit the personal config file
import subprocess
REPO = '/content/project'
BRANCH = 'btt_setup_VD'

for cmd in [
    ['git', 'checkout', BRANCH],
    ['git', 'add', 'code/configs/phase1_config_vd.yaml'],
    ['git', 'commit', '-m', 'Add personal config copy for fine-tuning experiments'],
    ['git', 'push', 'origin', BRANCH],
]:
    r = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO)
    out = (r.stdout + r.stderr).strip()
    if out:
        print(out)

Your branch is up to date with 'origin/btt_setup_VD'.
Already on 'btt_setup_VD'
[btt_setup_VD bd3c1aa] Add personal config copy for fine-tuning experiments
 1 file changed, 225 insertions(+)
 create mode 100644 code/configs/phase1_config_vd.yaml
To https://github.com/Break-Through-Tech/Automation-Anywhere-1B-domain-specific-theme-labeling-via-slm-distillation.git
   795bcb6..bd3c1aa  btt_setup_VD -> btt_setup_VD


In [11]:
#Commit the run_experiment.py
import subprocess
REPO = '/content/project'
BRANCH = 'btt_setup_VD'

for cmd in [
    ['git', 'add', 'code/run_experiments.py'],
    ['git', 'commit', '-m', 'Update experiment grid'],
    ['git', 'push', 'origin', BRANCH],
]:
    r = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO)
    print((r.stdout + r.stderr).strip())


[btt_setup_VD 08df09e] Update experiment grid
 1 file changed, 30 insertions(+), 28 deletions(-)
To https://github.com/Break-Through-Tech/Automation-Anywhere-1B-domain-specific-theme-labeling-via-slm-distillation.git
   0147211..08df09e  btt_setup_VD -> btt_setup_VD


# **Run Experiment**

In [16]:
!python "{CODE_DIR}/run_experiments.py"


Running experiment: baseline
[ERROR] Experiment 'baseline' failed:
/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:1551: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *
Traceback (most recent call last):
  File "/content/project/code/main.py", line 337, in <module>
    main()
    ~~~~^^
  File "/content/project/code/main.py", line 322, in main
    run_phase1(cfg)
    ~~~~~~~~~~^^^^^
  File "/content/project/code/phase1/pipeline.py", line 176, in run_phase1
    _, tokenizer_tmp = load_model_and_tokenizer(cfg)
                       ~~~~~~~~~~~~~~~~~~~~~~~~^^^^^
  File "/content/project/code/phase1/finetuning/trainer.py", line 51, in load_model_and_tokenizer
    model, tokenizer = _load_colab(model_id, cfg)
                    

In [ ]:
import pandas as pd
results = pd.read_csv(f'{CODE_DIR}/experiment_results.csv')
results

commit_file(
    source_path=f'{CODE_DIR}/experiment_results.csv',
    dest_relative_path='experiment_results.csv',
    commit_message='Add experiment results from epoch/rank/lr sweep',
    under_code_dir=True,
)

SameFileError: '/content/project/code/experiment_results.csv' and '/content/project/code/experiment_results.csv' are the same file